[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FrontierDevelopmentLab/2024-HL-Thermo-CL/blob/main/public/inference_forecast_example.ipynb)

# Karman TFT Forecasting Model — Inference Demo

**Authors:** William Fawcett, James Walsh, Giacomo Acciarini, Aishwarya Kumar, Jordi Vila Perez

This notebook walks through running inference with a pre-trained **Temporal Fusion Transformer (TFT)** that forecasts thermospheric neutral density (kg/m³) from space-weather time series and satellite orbital parameters.

> **Self-contained**: This notebook runs entirely in Google Colab with no local dependencies — the model weights and sample data are downloaded from a public S3 bucket at runtime.

## How it works

The TFT ingests **temporal context** — a ~7-day history of space-weather observations — to produce its forecast. It processes three input channels:

| Input channel | Shape | Description |
|--------------|-------|-------------|
| **Static** | (8,) | Satellite orbit params: altitude, latitude, lon sin/cos, day-of-year sin/cos, time-of-day sin/cos |
| **Historical** | (100, 25) | 100 time steps (×100 min ≈ 7 days) of OMNI indices, OMNI magnetic field, OMNI solar wind, SOHO EUV irradiance, and NRLMSISE-00 model densities |
| **Future** | (1, 1) | NRLMSISE-00 density at the forecast time (context for the decoder) |

The TFT architecture uses **variable selection networks** to learn which features matter, **LSTMs** for sequential processing, and **multi-head attention** over the time dimension to capture long-range dependencies. The output is a quantile prediction (here, the median) in normalized log-space, which is converted back to physical density.

$$\rho_{\text{pred}} = 10^{\;\frac{(\log_{\max} - \log_{\min})(\hat{y} + 1)}{2} \;+\; \log_{\min}}$$

## 1. Install dependencies

In [ ]:
!pip install -q torch omegaconf tft-torch matplotlib

## 2. Imports and model constants

These constants must match the values used during training. The model was trained with:
- A **10,000-minute lookback** (~7 days) sampled every **100 minutes** → 100 historical time steps
- **25 historical features** drawn from five data sources
- **8 static features** encoding the satellite’s orbital position
- **1 future feature** (NRLMSISE-00 density at the target time)

In [ ]:
import io
import torch
import torch.nn as nn
import torch.nn.init as init
from urllib.request import urlopen
from omegaconf import OmegaConf
from tft_torch import tft
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float32)

# Model architecture (must match training)
NUM_STATIC_NUMERIC = 8
NUM_HISTORICAL_NUMERIC = 25
NUM_FUTURE_NUMERIC = 1
NUM_HISTORICAL_STEPS = 100  # lag_minutes=10000 / resolution_minutes=100

# Density normalization bounds (log10 space, from training data)
LOG_MIN = -13.99995231628418
LOG_MAX = -9.80734920501709

print(f"Historical window: {NUM_HISTORICAL_STEPS} steps \u00d7 100 min = {NUM_HISTORICAL_STEPS * 100 / 60:.0f} hours")
print(f"Log-density range: [{LOG_MIN:.4f}, {LOG_MAX:.4f}]")

## 3. Define the scaling functions

Thermospheric density spans many orders of magnitude (~10⁻¹⁴ to 10⁻¹⁰ kg/m³). The model works in **normalized log-space**, mapping densities to $[-1, 1]$ via:

$$\text{scaled} = 2 \cdot \frac{\log_{10}(\rho) - \log_{\min}}{\log_{\max} - \log_{\min}} - 1$$

and the inverse:

$$\rho = 10^{\;\frac{(\log_{\max} - \log_{\min})(\text{scaled} + 1)}{2} \;+\; \log_{\min}}$$

In [ ]:
def scale_density(density, log_min, log_max):
    """Map physical density (kg/m\u00b3) into normalized log-space [-1, 1]."""
    tmp = torch.log10(density)
    return 2.0 * (tmp - log_min) / (log_max - log_min) - 1.0


def unscale_density(scaled, log_min, log_max):
    """Map normalized log-space value back to physical density (kg/m\u00b3)."""
    tmp = (log_max - log_min) * (scaled + 1) / 2 + log_min
    return torch.pow(10, tmp)


# Quick sanity check: round-trip a known density value
test_rho = torch.tensor(1.5e-12)
reconstructed = unscale_density(scale_density(test_rho, LOG_MIN, LOG_MAX), LOG_MIN, LOG_MAX)
print(f"Original:      {test_rho.item():.6e}")
print(f"Round-tripped:  {reconstructed.item():.6e}")
print(f"Match: {torch.allclose(test_rho, reconstructed)}")

## 4. Weight initialization

The model weights are initialized before loading the checkpoint. This uses Xavier normal initialization for linear layers and orthogonal initialization for LSTMs — the same scheme used during training.

In [ ]:
def weight_init(m):
    """Initialize model weights (must match training)."""
    if isinstance(m, nn.Linear):
        init.xavier_normal_(m.weight.data)
        if m.bias is not None:
            init.normal_(m.bias.data)
    elif isinstance(m, (nn.LSTM, nn.LSTMCell)):
        for param in m.parameters():
            if len(param.shape) >= 2:
                init.orthogonal_(param.data)
            else:
                init.normal_(param.data)
    elif isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):
        init.normal_(m.weight.data, mean=1, std=0.02)
        init.constant_(m.bias.data, 0)

## 5. Load sample data from S3

The sample data contains **100 real observations** extracted from the [Karman dataset](https://github.com/FrontierDevelopmentLab/2024-HL-Thermo-CL), pre-processed with the same pipeline used during training (QuantileTransformer for OMNI/SOHO, min-max scaling for static features, log-space scaling for NRLMSISE-00).

The 25 historical features are drawn from five data sources:

| Source | Features | Count |
|--------|----------|-------|
| OMNI Indices | AE, AL, AU, SYM-D, SYM-H, ASY-D | 6 |
| OMNI Magnetic Field | Bx, By, Bz (GSE) | 3 |
| OMNI Solar Wind | Speed, Vx, Vy, Vz | 4 |
| SOHO EUV Irradiance | 30 nm, 25 nm | 2 |
| NRLMSISE-00 Densities | 10 lat/lon grid points | 10 |
| | **Total** | **25** |

In [ ]:
SAMPLE_DATA_URL = (
    "https://nasa-radiant-data.s3.amazonaws.com/"
    "helioai-datasets/hl-therm/models/sample_inputs_tft.pt"
)

print("Downloading sample data from S3 ...")
with urlopen(SAMPLE_DATA_URL) as resp:
    data_bytes = resp.read()
print(f"Downloaded {len(data_bytes) / 1024:.1f} KB")

sample_data = torch.load(io.BytesIO(data_bytes), map_location='cpu', weights_only=False)

# Unpack
static_feats = sample_data['static_feats_numeric']
historical_ts = sample_data['historical_ts_numeric']
future_ts = sample_data['future_ts_numeric']
ground_truth = sample_data['ground_truth']
nrlmsise00 = sample_data['nrlmsise00']
dates = sample_data['dates']

print(f"\nLoaded {static_feats.shape[0]} samples")
print(f"  static_feats_numeric:  {static_feats.shape}")
print(f"  historical_ts_numeric: {historical_ts.shape}")
print(f"  future_ts_numeric:     {future_ts.shape}")
print(f"\nDate range: {dates[0]} -> {dates[-1]}")

## 6. Build and load the TFT model

The Temporal Fusion Transformer is configured via an OmegaConf dictionary. Key architecture choices:

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `state_size` | 64 | Width of all internal embeddings and GRN layers |
| `lstm_layers` | 2 | Depth of the encoder and decoder LSTMs |
| `attention_heads` | 4 | Number of heads in the interpretable multi-head attention |
| `dropout` | 0.05 | Applied throughout for regularization |
| `output_quantiles` | [0.5] | Predict the median (50th percentile) |

The model has **1,074,865 trainable parameters** and achieves ~14.9% MAPE on the validation set.

In [ ]:
MODEL_URL = (
    "https://nasa-radiant-data.s3.amazonaws.com/"
    "helioai-datasets/hl-therm/models/"
    "ts_karman_model_tft_run_gpu_tft_w_omni_and_soho_wo_indices_and_proxies"
    "_w_10000_lag_100_resolution_valid_mape_14.936_params_1074865.torch"
)

# Build model configuration
configuration = OmegaConf.create({
    'model': {
        'dropout': 0.05,
        'state_size': 64,
        'output_quantiles': [0.5],
        'lstm_layers': 2,
        'attention_heads': 4,
    },
    'task_type': 'regression',
    'target_window_start': None,
    'data_props': {
        'num_historical_numeric': NUM_HISTORICAL_NUMERIC,
        'num_static_numeric': NUM_STATIC_NUMERIC,
        'num_future_numeric': NUM_FUTURE_NUMERIC,
    },
})

# Instantiate and initialize
model = tft.TemporalFusionTransformer(configuration)
model.apply(weight_init)

# Download weights from S3
print("Downloading model weights from S3 ...")
with urlopen(MODEL_URL) as resp:
    model_bytes = resp.read()
print(f"Downloaded {len(model_bytes) / 1024:.1f} KB")

model.load_state_dict(torch.load(io.BytesIO(model_bytes), map_location='cpu', weights_only=True))
model.eval()

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel loaded: {n_params:,} trainable parameters")

## 7. Run inference

The TFT forward pass takes a dictionary of the three input channels and returns `predicted_quantiles` with shape `(batch_size, future_steps, num_quantiles)`. Since we configured a single quantile (the median) and a single future step, the output is `(N, 1, 1)` which we squeeze to a 1-D tensor.

In [ ]:
inputs = {
    'static_feats_numeric':  static_feats,
    'historical_ts_numeric': historical_ts,
    'future_ts_numeric':     future_ts,
}

with torch.no_grad():
    out = model(inputs)

# Extract median prediction and convert to physical density
predicted_quantiles = out['predicted_quantiles']  # (N, 1, 1)
median_pred = predicted_quantiles[:, :, 0].squeeze()  # (N,)
density_pred = unscale_density(median_pred, LOG_MIN, LOG_MAX)

print(f"Output shape: {predicted_quantiles.shape}")
print(f"Prediction range: {density_pred.min().item():.3e} \u2013 {density_pred.max().item():.3e} kg/m\u00b3")

## 8. Results

We compare the TFT predictions against:
- **Ground truth**: actual thermospheric density from TU Delft satellite observations
- **NRLMSISE-00**: the standard empirical atmosphere model (a common baseline)

Accuracy is measured using **Absolute Percentage Error (APE)** per sample, and **Mean APE (MAPE)** across all samples.

In [ ]:
N_DISPLAY = 20  # number of samples to show in the table

batch_size = static_feats.shape[0]
nn_apes = []
msise_apes = []

for i in range(batch_size):
    pred_val = density_pred[i].item()
    true_val = ground_truth[i].item()
    msise_val = nrlmsise00[i].item()
    nn_apes.append(abs(pred_val - true_val) / abs(true_val) * 100)
    msise_apes.append(abs(msise_val - true_val) / abs(true_val) * 100)

print(f"Showing first {N_DISPLAY} of {batch_size} samples\n")
print(f"{'#':>3}  {'Date':>22}  {'Predicted':>14}  {'Truth':>14}  {'NRLMSISE':>14}  {'TFT err':>8}  {'MSISE err':>9}")
print("-" * 95)
for i in range(min(N_DISPLAY, batch_size)):
    print(f"{i:>3}  {dates[i]:>22}  {density_pred[i].item():>14.4e}  {ground_truth[i].item():>14.4e}  {nrlmsise00[i].item():>14.4e}  {nn_apes[i]:>6.1f}%  {msise_apes[i]:>7.1f}%")
if batch_size > N_DISPLAY:
    print(f"  ... ({batch_size - N_DISPLAY} more rows)")

print("-" * 95)
print(f"\nMean Absolute Percentage Error (all {batch_size} samples):")
print(f"  TFT model:   {sum(nn_apes)/len(nn_apes):.1f}%")
print(f"  NRLMSISE-00: {sum(msise_apes)/len(msise_apes):.1f}%")

## 9. Visualization: Predicted vs. Ground Truth

A perfect model would place all points on the diagonal. We also show the NRLMSISE-00 baseline to illustrate the improvement from the TFT. Note the log-scale axes — densities span several orders of magnitude.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

gt_vals = [ground_truth[i].item() for i in range(batch_size)]
pred_vals = [density_pred[i].item() for i in range(batch_size)]
msise_vals = [nrlmsise00[i].item() for i in range(batch_size)]

ax.scatter(gt_vals, msise_vals, alpha=0.6, label='NRLMSISE-00 (baseline)',
           marker='x', color='gray', s=60)
ax.scatter(gt_vals, pred_vals, alpha=0.8, label='TFT prediction',
           marker='o', color='tab:blue', edgecolors='k', linewidths=0.3, s=60)

# Perfect-prediction diagonal
all_vals = gt_vals + pred_vals + msise_vals
lo = min(all_vals) * 0.5
hi = max(all_vals) * 2.0
ax.plot([lo, hi], [lo, hi], 'k--', lw=1, label='Perfect prediction')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Ground truth density [kg/m\u00b3]')
ax.set_ylabel('Predicted density [kg/m\u00b3]')
ax.set_title('TFT Predicted vs. Ground Truth Density')
ax.legend()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 10. Visualization: Per-sample error comparison

Side-by-side bars show the percentage error of the TFT model versus the NRLMSISE-00 baseline for each sample.

In [ ]:
import numpy as np

n_plot = min(N_DISPLAY, batch_size)
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(n_plot)
width = 0.35

bars1 = ax.bar(x - width/2, nn_apes[:n_plot], width,
               label=f'TFT (mean {sum(nn_apes)/len(nn_apes):.1f}%)',
               color='tab:blue', edgecolor='k', linewidth=0.3)
bars2 = ax.bar(x + width/2, msise_apes[:n_plot], width,
               label=f'NRLMSISE-00 (mean {sum(msise_apes)/len(msise_apes):.1f}%)',
               color='gray', edgecolor='k', linewidth=0.3, alpha=0.7)

ax.axhline(sum(nn_apes)/len(nn_apes), color='tab:blue', ls='--', lw=1.5, alpha=0.5)
ax.axhline(sum(msise_apes)/len(msise_apes), color='gray', ls='--', lw=1.5, alpha=0.5)

ax.set_xlabel('Sample index')
ax.set_ylabel('Absolute Percentage Error [%]')
ax.set_title(f'Per-Sample Error: TFT vs NRLMSISE-00 (first {n_plot} of {batch_size})')
ax.set_xticks(x)
ax.legend()
plt.tight_layout()
plt.show()